# **Cómo funciona optuna**

Funciona con 3 conceptos clave, un **study**, que viene a ser el experimento completo, **trials**, cada evaluación de hiperparámetros, y una **función objetivo** (la función que se intenta minimizar/maximizar).

In [1]:
%pip install optuna
%pip install optuna-dashboard plotly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 1.3 MB/s  0:00:02 eta 0:00:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.4/613.4 kB 1.1 MB/s  0:00:0036m-:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [optuna]2m6/7 [optuna]]my]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 2.6 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 3.2 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [optuna-dashboard][optuna-dashboard]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn # Modulo para redes neuronales
import torch.optim as optim # Modulo para optimizadores
import optuna # Modulo para optimización de hiperparámetros
from optuna.pruners import MedianPruner

# Modulo para cargar y transformar datos
import torchvision
from torchvision import datasets, transforms, models

# Modulo para visualización
import matplotlib.pyplot as plt
import numpy as np

# Modulo para evaluación de modelos
from sklearn.metrics import classification_report, confusion_matrix

import os

/home/franco/Redes-Neuronales-en-Keras/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

# **Carga de los datos**

In [6]:
raiz = '../../data/Ojos'

# Datos de entrenamiento
train_dir = os.path.join(raiz, 'train')
train_dir_normal = os.path.join(train_dir, 'Normales')
train_dir_glaucoma = os.path.join(train_dir, 'Glaucomas')

# Datos de validación
val_dir = os.path.join(raiz, 'val')
val_dir_normal = os.path.join(val_dir, 'Normales')
val_dir_glaucoma = os.path.join(val_dir, 'Glaucomas')

# Datos de prueba
test_dir = os.path.join(raiz, 'test')
test_dir_normal = os.path.join(test_dir, 'Normales')
test_dir_glaucoma = os.path.join(test_dir, 'Glaucomas')

# Muestro la cantidad de imágenes en cada conjunto
suma_entrenamiento = len(os.listdir(train_dir_normal)) + len(os.listdir(train_dir_glaucoma))
suma_validacion = len(os.listdir(val_dir_normal)) + len(os.listdir(val_dir_glaucoma))
suma_prueba = len(os.listdir(test_dir_normal)) + len(os.listdir(test_dir_glaucoma))

print("Datos de entrenamiento:", suma_entrenamiento, "--- Normales:", len(os.listdir(train_dir_normal)), "--- Glaucoma:", len(os.listdir(train_dir_glaucoma)))
print("Datos de validación:", suma_validacion, "--- Normales:", len(os.listdir(val_dir_normal)), "--- Glaucoma:", len(os.listdir(val_dir_glaucoma)))
print("Datos de prueba:", suma_prueba, "--- Normales:", len(os.listdir(test_dir_normal)), "--- Glaucoma:", len(os.listdir(test_dir_glaucoma)))

Datos de entrenamiento: 339 --- Normales: 219 --- Glaucoma: 120
Datos de validación: 48 --- Normales: 31 --- Glaucoma: 17
Datos de prueba: 98 --- Normales: 63 --- Glaucoma: 35


# **Data Augmentation**

In [7]:
# Data augmentation para el conjunto de entrenamiento
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # Volteo horizontal aleatorio
    transforms.RandomVerticalFlip(), # Volteo vertical aleatorio
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(1.0, 1.0)), # Zoom aleatorio
    transforms.RandomRotation(20), # Rotación aleatoria
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Para el conjunto de validacion
val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Para el conjunto de prueba
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [8]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transforms)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transforms)
test_dataset = datasets.ImageFolder(test_dir, transform=test_transforms)

In [9]:
train_loader = torch.utils.data.DataLoader(train_dataset,
                                           batch_size=16,
                                           shuffle=True)

val_loader = torch.utils.data.DataLoader(val_dataset,
                                         batch_size=16,
                                         shuffle=False)

test_loader = torch.utils.data.DataLoader(test_dataset,
                                          batch_size=1,
                                          shuffle=False)

# **Definición de la función objetivo**

In [12]:
def objective(trial):
    # ── Hiperparámetros a explorar ──────────────────────────────────────────
    lr          = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    batch_size  = trial.suggest_categorical("batch_size", [4, 8])
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW"])
    weight_decay   = trial.suggest_float("weight_decay", 1e-5, 1e-2, log=True)

    # ── DataLoaders con el batch_size del trial ─────────────────────────────
    train_loader_t = torch.utils.data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )
    val_loader_t = torch.utils.data.DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False
    )

    # ── Modelo: cargamos VGG19 fresco en cada trial ─────────────────────────
    model = models.vgg19(pretrained=True)
    model.classifier[6] = nn.Linear(4096, 2)

    # Fine-tuning: solo entrenamos el clasificador
    for param in model.features.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True

    model.to(device)

    # ── Optimizador ─────────────────────────────────────────────────────────
    optimizer_cls = getattr(optim, optimizer_name)
    optimizer_t = optimizer_cls(
        model.parameters(), lr=lr, weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    # ── Entrenamiento corto para buscar hiperparámetros (10 épocas) ─────────
    # No necesitas 100 épocas aquí, Optuna solo necesita una señal relativa
    N_EPOCHS_SEARCH = 10
    best_val_loss = float("inf")

    for epoch in range(N_EPOCHS_SEARCH):
        # Entrenamiento
        model.train()
        for images, labels in train_loader_t:
            images, labels = images.to(device), labels.to(device)
            optimizer_t.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer_t.step()

        # Validación
        model.eval()
        val_loss_epoch = 0.0
        with torch.no_grad():
            for images, labels in val_loader_t:
                images, labels = images.to(device), labels.to(device)
                val_loss_epoch += criterion(model(images), labels).item()

        val_loss_epoch /= len(val_loader_t)

        if val_loss_epoch < best_val_loss:
            best_val_loss = val_loss_epoch

        print(f"Trial {trial.number} - Epoch {epoch+1}/{N_EPOCHS_SEARCH} - Val Loss: {val_loss_epoch:.4f}")

        # Pruning: cancela trials poco prometedores temprano ✂️
        trial.report(val_loss_epoch, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return best_val_loss

Una vez definida la función objetivo la llamamos para que haga las pruebas.

In [13]:
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=MedianPruner(n_warmup_steps=3),  # no poda antes de la época 3
    # storage="sqlite:///glaucoma_optuna.db",  # descomenta para persistencia
    # study_name="vgg19_glaucoma",
    # load_if_exists=True,
)

study.optimize(objective, n_trials=10)  # 20 trials suele ser suficiente para estos params

print("\n── Mejores hiperparámetros ──────────────────")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"  → Val loss: {study.best_value:.4f}")

[I 2026-05-05 18:42:11,008] A new study created in memory with name: no-name-f8663be0-47df-47c1-9b6d-c896a41ddd76
/home/franco/Redes-Neuronales-en-Keras/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/franco/Redes-Neuronales-en-Keras/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Trial 0 - Epoch 1/10 - Val Loss: 0.6183
Trial 0 - Epoch 2/10 - Val Loss: 0.7760
Trial 0 - Epoch 3/10 - Val Loss: 0.6988
Trial 0 - Epoch 4/10 - Val Loss: 0.6167
Trial 0 - Epoch 5/10 - Val Loss: 0.7935
Trial 0 - Epoch 6/10 - Val Loss: 0.5602
Trial 0 - Epoch 7/10 - Val Loss: 0.5910
Trial 0 - Epoch 8/10 - Val Loss: 0.5903
Trial 0 - Epoch 9/10 - Val Loss: 0.5307


[I 2026-05-05 18:44:50,511] Trial 0 finished with value: 0.5306580468701819 and parameters: {'lr': 5.6115164153345e-05, 'batch_size': 4, 'optimizer': 'Adam', 'weight_decay': 2.9375384576328295e-05}. Best is trial 0 with value: 0.5306580468701819.


Trial 0 - Epoch 10/10 - Val Loss: 0.9359
Trial 1 - Epoch 1/10 - Val Loss: 0.6324
Trial 1 - Epoch 2/10 - Val Loss: 0.6331
Trial 1 - Epoch 3/10 - Val Loss: 0.5766
Trial 1 - Epoch 4/10 - Val Loss: 0.5507
Trial 1 - Epoch 5/10 - Val Loss: 0.5070
Trial 1 - Epoch 6/10 - Val Loss: 0.4896
Trial 1 - Epoch 7/10 - Val Loss: 0.6074
Trial 1 - Epoch 8/10 - Val Loss: 0.5670
Trial 1 - Epoch 9/10 - Val Loss: 0.6994


[I 2026-05-05 18:47:27,858] Trial 1 finished with value: 0.489584061006705 and parameters: {'lr': 1.3066739238053272e-05, 'batch_size': 4, 'optimizer': 'Adam', 'weight_decay': 0.008123245085588688}. Best is trial 1 with value: 0.489584061006705.


...

Trial 7 - Epoch 10/10 - Val Loss: 0.5685
Trial 8 - Epoch 1/10 - Val Loss: 0.7410
Trial 8 - Epoch 2/10 - Val Loss: 0.7364
Trial 8 - Epoch 3/10 - Val Loss: 0.9900


[I 2026-05-05 19:02:28,198] Trial 8 pruned. 


Trial 8 - Epoch 4/10 - Val Loss: 0.8004
Trial 9 - Epoch 1/10 - Val Loss: 0.6750
Trial 9 - Epoch 2/10 - Val Loss: 0.7444
Trial 9 - Epoch 3/10 - Val Loss: 0.8803
Trial 9 - Epoch 4/10 - Val Loss: 0.6117
Trial 9 - Epoch 5/10 - Val Loss: 0.7979
Trial 9 - Epoch 6/10 - Val Loss: 0.7666
Trial 9 - Epoch 7/10 - Val Loss: 0.7015
Trial 9 - Epoch 8/10 - Val Loss: 0.7748
Trial 9 - Epoch 9/10 - Val Loss: 0.6400


[I 2026-05-05 19:05:04,836] Trial 9 finished with value: 0.6116634793579578 and parameters: {'lr': 0.0001569639638866114, 'batch_size': 4, 'optimizer': 'Adam', 'weight_decay': 9.46217535646148e-05}. Best is trial 1 with value: 0.489584061006705.


Trial 9 - Epoch 10/10 - Val Loss: 0.8049

── Mejores hiperparámetros ──────────────────
  lr: 1.3066739238053272e-05
  batch_size: 4
  optimizer: Adam
  weight_decay: 0.008123245085588688
  → Val loss: 0.4896
